# IRIS on Kaggle: Phase 1 runs (M4 leakage canary, M1 baseline, M6 circadian recovery)

**Scope.** Only OPEN datasets (HUPA-UCM, BrisT1D-Open) may ever be attached here. DiaTrend, T1DEXI and OhioT1DM are controlled-access and must NOT be uploaded to Kaggle.
**Compute is light** (each experiment finishes in about a minute): Kaggle is used for an independent, repeatable run, not for speed. Accelerator: None.
**Before running:** the tagged repository release must already contain the verified `data/sources_registry.csv`, `data/manifest.csv` and AUDITED `configs/mappings/*.yaml` for the datasets below. This notebook never audits, maps or verifies anything; it only re-runs what was prepared and signed off.
**Nothing here is a result until the outputs are imported and verified (`scripts/verify_run_dir.py`) and judged against the validation checklist.**

In [ ]:
# ---- the ONLY cell you edit -------------------------------------------------------------
REPO_URL = "https://github.com/YOUR-USER/iris.git"   # your repository
TAG      = "v0.1.0"                                   # an immutable release tag, never a branch
# dataset id -> name of the attached Kaggle dataset folder under /kaggle/input/
DATASETS = {"hupa_ucm": "iris-hupa-ucm", "brist1d": "iris-brist1d-open"}
# ------------------------------------------------------------------------------------------

In [ ]:
import subprocess, sys, os, shutil, pathlib, json, platform
def sh(cmd, check=True):
    print("$", cmd); r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-4000:], r.stderr[-2000:])
    if check and r.returncode: raise SystemExit(f"command failed ({r.returncode}): {cmd}")
    return r
os.chdir("/kaggle/working")
sh(f"rm -rf iris && git clone {REPO_URL} iris")
os.chdir("iris")
sh(f"git checkout --detach {TAG}")                       # a tag, so the run id carries a clean commit hash
sh("git status --porcelain")                              # must print nothing
sh(f"{sys.executable} -m pip install -q -e . ")
print(platform.python_version())

In [ ]:
# copy the attached OPEN datasets into the repository layout; names must stay exactly as downloaded
for ds, folder in DATASETS.items():
    src = pathlib.Path("/kaggle/input") / folder
    assert src.exists(), f"attach the Kaggle dataset '{folder}' to this notebook (Add Input)"
    dst = pathlib.Path("data/raw") / ds
    dst.mkdir(parents=True, exist_ok=True)
    for f in src.rglob("*"):
        if f.is_file():
            t = dst / f.relative_to(src); t.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(f, t)
    print(ds, sum(1 for _ in dst.rglob('*') if _.is_file()), "files")

In [ ]:
# production ingest re-checks every raw file against the committed manifest hashes and refuses anything unregistered/unverified
for ds in DATASETS:
    sh(f"{sys.executable} -m iris.tools.ingest_dataset {ds} --mode production")
sh(f"{sys.executable} -m iris.tools.ingest_dataset {list(DATASETS)[-1]} --mode production --combine")
sh(f"{sys.executable} -m iris.tools.validate_person_day data/processed/person_day.parquet")

In [ ]:
# BLOCKED is a legitimate outcome (the repository refuses to invent an input); it is printed, never hidden
for exp in ("M4_leakage", "M1_baseline", "M6_circadian_recovery"):
    sh(f"{sys.executable} -m iris.tools.run_experiment {exp} --mode production", check=False)

In [ ]:
# package everything needed to bring the runs back; NO raw data is included
os.makedirs("/kaggle/working/out", exist_ok=True)
sh(f"{sys.executable} -m pip freeze > /kaggle/working/out/pip_freeze.txt")
meta = {"repo": REPO_URL, "tag": TAG, "python": platform.python_version(), "platform": platform.platform(),
        "commit": subprocess.run("git rev-parse HEAD", shell=True, capture_output=True, text=True).stdout.strip()}
json.dump(meta, open("/kaggle/working/out/environment.json", "w"), indent=2)
shutil.copytree("results/runs", "/kaggle/working/out/runs", dirs_exist_ok=True)
shutil.copytree("data/processed", "/kaggle/working/out/processed_qc", dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("*.parquet"))          # QC reports only, no derived person-level data
shutil.make_archive("/kaggle/working/iris_kaggle_outputs", "zip", "/kaggle/working/out")
print("download: iris_kaggle_outputs.zip from the Output tab")